In [3]:
import pandas as pd
import os
from dotenv import load_dotenv
from openai import OpenAI
import json
load_dotenv(override=True)


# ==========================
# OpenAI Configuration
# ==========================

config_str = os.getenv("GPT_LUNA_CONFIG")

if config_str is None:
    raise ValueError("model environment variable is missing")

config = json.loads(config_str)

MODEL_NAME = config["name"]
REASONING_EFFORT = config["reasoning_effort"]

print(f"Using model: {MODEL_NAME} with reasoning effort: {REASONING_EFFORT}")

OPENAI_API_KEY = os.getenv(
    "OPENAI_API_KEY"
)

SLEEP_BETWEEN_CALLS = int(
    os.getenv("SLEEP_BETWEEN_CALLS", "1")
)


client = OpenAI(
    api_key=OPENAI_API_KEY
)

DATA_PATH = os.getenv("DATA_PATH", "../data/")

df=pd.read_csv(f'{DATA_PATH}df_n3.csv')



Using model: gpt-5.6-luna with reasoning effort: low


In [2]:
import json
import time
import random
import pandas as pd
from collections import defaultdict
from collections import Counter


# =========================
# PROMPT BUILDING
# =========================

def build_dedup_messages(rows, shuffle=False):
    """
    rows: list of dicts for comments that share the SAME patch_id.
          each dict must have: comment_id, generated_comment, 
        #   category, severity,
          hunk, generated_context
    """
    code_patch = rows[0]
    comments = list(rows)
    if shuffle:
        random.shuffle(comments)

    comments_text = "\n\n".join(
        f'ID: {c["comment_id"]}\n'
        f'Comment: {c["generated_comment"]}'
        # f'Category: {c["category"]}\n'
        # f'Severity: {c["severity"]}\n'
        for c in comments
    )

    system_prompt = """
You are an expert software engineer reviewing a set of code-review comments that were independently generated by different models for the SAME code change.

INPUT:
You may receive:
- Pull request title
- Target file
- Related code hunks from the same file
- Surrounding code context, wrapped in <context> tags
- The code patch, wrapped in <patch> tags
- A list of generated review comments, each with its ID, category, and severity

Use the available context only to understand the patch and determine whether two comments refer to the same underlying issue. Do not group comments simply because they mention nearby code or similar identifiers.

TASK 1 - GROUPING:
Group comments that describe the SAME underlying issue, even if phrased differently or focused on different symptoms. Two comments belong in the same group ONLY IF a human reviewer would consider one redundant given the other.

Do NOT group comments together just because they:
- refer to the same lines, variables, or functions but raise different concerns (e.g. bug vs. refactoring suggestion).
- use similar wording but identify different root causes.

Every comment must appear in exactly one group. A comment with no duplicate forms its own group.

TASK 2 - SYNTHESIS:
For every group containing TWO OR MORE comments, generate a single "synthesis_comment" that represents their shared concern. It should:
- Capture only the common issue.
- Be concise.
- Read like a human review comment, not a summary of comments.
- Avoid copying any individual comment verbatim.

Do NOT include "synthesis_comment" for singleton groups.

OUTPUT (STRICT JSON ONLY):
{
  "groups": [
    {
      "comment_ids": ["<id1>", "<id2>"],
      "issue_type": "short label",
      "target": "variable/function/line",
      "synthesis_comment": "..."
    },
    {
      "comment_ids": ["<id3>"],
      "issue_type": "short label",
      "target": "..."
    }
  ]
}

Use ONLY the exact comment IDs provided. Do not invent, split, merge, or omit IDs.
"""
    user_prompt = ""

    if "pr_title" in code_patch and pd.notna(code_patch["pr_title"]):

        user_prompt += f"""
PULL REQUEST TITLE:
{code_patch["pr_title"]}

"""

    if "target_file" in code_patch and pd.notna(code_patch["target_file"]):

        user_prompt += f"""
TARGET FILE:
{code_patch["target_file"]}

"""
    if "relevant_same_file_code_hunks" in code_patch and pd.notna(code_patch["relevant_same_file_code_hunks"]):

        user_prompt += f"""
RELATED CODE HUNKS IN TARGET FILE:
{code_patch["relevant_same_file_code_hunks"]}

"""

    if "relevant_context" in code_patch and pd.notna(code_patch["relevant_context"]):

        if str(code_patch["relevant_context"]).strip():

            user_prompt += """
SURROUNDING CODE CONTEXT:
This is additional context extracted from the original file around the changed code to help you in problem identification.
<context>
"""

            user_prompt += code_patch["relevant_context"]
            user_prompt += "\n</context>\n\n"

    user_prompt += f"""
CODE PATCH:
<patch>
{code_patch["hunk"]}
</patch>

"""
    user_prompt += f"""
COMMENTS TO GROUP:
{comments_text}
"""

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]




N_CONSISTENCY_RUNS = 3


def build_batch_jsonl(df, output_path="artifacts/dedup_batch.jsonl",
                      n_runs=N_CONSISTENCY_RUNS):
    """
    Build a JSONL file containing one Batch API request per LLM call.

    Each line corresponds to one consistency run for one patch.

    For each patch:
        run 0 -> original comment order
        run 1 -> shuffled comments
        run 2 -> shuffled comments

    The generated JSONL can then be submitted to the OpenAI Batch API.
    """

    required_cols = [
        "comment_id",
        "generated_comment",
        "category",
        "generation_system",
        "patch_id",
        "severity",
        "hunk"
    ]

    for c in required_cols:
        if c not in df.columns:
            raise ValueError(f"Missing required column: {c}")

    # Create output directory if needed
    import os
    os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)

    patch_groups = df.groupby("patch_id")

    total_requests = 0

    with open(output_path, "w", encoding="utf-8") as f:

        for patch_id, group in patch_groups:

            rows = group.to_dict("records")

            # Make sure optional fields exist
            for r in rows:
                r.setdefault("generated_context", "")

            # One request for each consistency run
            for run_idx in range(n_runs):

                # Run 0 keeps original order.
                # Later runs shuffle the comments.
                shuffle = run_idx > 0

                messages = build_dedup_messages(
                    rows,
                    shuffle=shuffle
                )

                batch_request = {
                    "custom_id": f"dedup_{patch_id}_run_{run_idx}",
                    "method": "POST",
                    "url": "/v1/chat/completions",
                    "body": {
                        "model": MODEL_NAME,
                        "messages": messages,
                        "response_format": {
                            "type": "json_object"
                        }
                    }
                }

                # Add reasoning effort only if you use it
                if REASONING_EFFORT:
                    batch_request["body"]["reasoning_effort"] = REASONING_EFFORT
                else:
                    batch_request["body"]["temperature"] = 0.4

                # Write exactly ONE JSON object per line
                f.write(
                    json.dumps(
                        batch_request,
                        ensure_ascii=False
                    ) + "\n"
                )

                total_requests += 1

    print(f"[DONE] JSONL created: {output_path}")
    print(f"[DONE] Number of patches: {patch_groups.ngroups}")
    print(f"[DONE] Number of requests: {total_requests}")

    return output_path

In [4]:
build_batch_jsonl(
    df,
    output_path="dedup_ready_batch_50.jsonl"
)

[DONE] JSONL created: dedup_ready_batch_50.jsonl
[DONE] Number of patches: 50
[DONE] Number of requests: 150


'dedup_ready_batch_50.jsonl'